# SQLite Checkpointer — What's Actually Stored?

`SqliteSaver` writes LangGraph state to a local `.db` file using three tables.
This notebook runs a real conversation, then opens the database and shows
every row, every column, and what each byte means.

```
checkpoints.db
├── checkpoints        ← one row per super-step  (the snapshots)
├── checkpoint_writes  ← one row per pending node write (fault tolerance)
└── checkpoint_migrations  ← internal version tracking
```

> **No server needed.** SQLite is a file — run this notebook anywhere.

## 0 · Setup

In [ ]:
import os, sqlite3
import pandas as pd
from IPython.display import display

_ENV = '/Users/datasense/Desktop/langgrapgh-agent/.env'
for line in open(_ENV):
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = 'checkpoints_demo.db'
print('Setup complete. DB will be created at:', DB_PATH)

## 1 · Build the graph with SqliteSaver

In [ ]:
llm = init_chat_model('openai:gpt-4o-mini', temperature=0)

def chat_node(state: MessagesState):
    return {'messages': [llm.invoke(state['messages'])]}

builder = StateGraph(MessagesState)
builder.add_node('chat', chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))
print('Graph compiled with SqliteSaver ->', DB_PATH)

## 2 · Run a 3-turn conversation

Each `invoke()` call creates new rows in `checkpoints`. Watch the table grow.

In [ ]:
alice = {'configurable': {'thread_id': 'alice'}}
bob   = {'configurable': {'thread_id': 'bob'}}

r1 = graph.invoke({'messages': [{'role': 'user', 'content': 'Hi! My name is Satvik and I love LangGraph.'}]}, alice)
print('Turn 1 (alice):', r1['messages'][-1].content[:70])

r2 = graph.invoke({'messages': [{'role': 'user', 'content': "What's my name and what do I love?"}]}, alice)
print('Turn 2 (alice):', r2['messages'][-1].content[:70])

r3 = graph.invoke({'messages': [{'role': 'user', 'content': 'What programming language do I probably use?'}]}, alice)
print('Turn 3 (alice):', r3['messages'][-1].content[:70])

r4 = graph.invoke({'messages': [{'role': 'user', 'content': "I'm Bob and I'm new here."}]}, bob)
print('Turn 1 (bob):  ', r4['messages'][-1].content[:70])

---
## 3 · Open the raw SQLite database

Let's see every table, every column, every row — exactly as SQLite stores them.

In [ ]:
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [r[0] for r in cur.fetchall()]
print('Tables in', DB_PATH, ':', tables)

for t in tables:
    cur.execute(f'PRAGMA table_info({t})')
    cols = cur.fetchall()
    print(f'\n  {t}')
    for col in cols:
        print(f'    col {col[1]:35s}  type={col[2]}')

### Table: `checkpoints`

| Column | Type | What it stores |
|--------|------|----------------|
| `thread_id` | TEXT | which conversation ("alice", "bob", ...) |
| `checkpoint_ns` | TEXT | subgraph namespace (empty = root graph) |
| `checkpoint_id` | TEXT | UUID for this specific snapshot |
| `parent_checkpoint_id` | TEXT | UUID of the previous snapshot (chain) |
| `type` | TEXT | serialisation format (`msgpack` or `json`) |
| `checkpoint` | BLOB | serialised full graph state (all channels) |
| `metadata` | BLOB | step number, source, node writes |

In [ ]:
df = pd.read_sql_query(
    'SELECT thread_id, checkpoint_ns, checkpoint_id, parent_checkpoint_id, type, '
    'length(checkpoint) as checkpoint_bytes, length(metadata) as metadata_bytes '
    'FROM checkpoints ORDER BY checkpoint_id',
    conn
)
print(f'checkpoints table: {len(df)} rows')
display(df)

### Decoding a checkpoint blob

The `checkpoint` column is a **serialised blob** (msgpack or JSON+). LangGraph's
`JsonPlusSerializer` can decode it back to the live Python state.

In [ ]:
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
serde = JsonPlusSerializer()

cur.execute(
    'SELECT checkpoint_id, type, checkpoint, metadata FROM checkpoints '
    'WHERE thread_id=? ORDER BY checkpoint_id DESC LIMIT 1',
    ('alice',)
)
row = cur.fetchone()
ckpt_id, typ, raw_ckpt, raw_meta = row

state   = serde.loads_typed((typ, raw_ckpt))
meta    = serde.loads_typed((typ, raw_meta))

print('checkpoint_id :', ckpt_id)
print('type          :', typ)
print('raw bytes     :', len(raw_ckpt), 'bytes')
print()
print('Decoded state keys :', list(state.keys()))
print('Step (from metadata):', meta.get('step'))
print('Source              :', meta.get('source'))
print('Node writes         :', list((meta.get('writes') or {}).keys()))

messages = state.get('channel_values', {}).get('messages', [])
print(f'\nMessages in this checkpoint: {len(messages)}')
for m in messages:
    role = getattr(m, 'type', '?')
    print(f'  [{role}] {str(getattr(m, "content", m))[:60]}')

### Table: `checkpoint_writes`

| Column | Type | What it stores |
|--------|------|----------------|
| `thread_id` | TEXT | same thread as parent checkpoint |
| `checkpoint_ns` | TEXT | subgraph namespace |
| `checkpoint_id` | TEXT | the checkpoint this write belongs to |
| `task_id` | TEXT | UUID of the node task that produced the write |
| `idx` | INTEGER | write order within the task |
| `channel` | TEXT | which state key was written (e.g. `messages`) |
| `type` | TEXT | serialisation format |
| `value` | BLOB | the serialised value written to that channel |

These rows let LangGraph recover from mid-step failures without re-running completed nodes.

In [ ]:
df_w = pd.read_sql_query(
    'SELECT thread_id, checkpoint_id, task_id, idx, channel, type, '
    'length(value) as value_bytes '
    'FROM checkpoint_writes ORDER BY checkpoint_id, idx',
    conn
)
print(f'checkpoint_writes table: {len(df_w)} rows')
display(df_w)

---
## 4 · LangGraph API view (same data, nicer interface)

`get_state` and `get_state_history` give you the decoded, structured view
of the same data you just saw as raw blobs above.

In [ ]:
snap = graph.get_state(alice)
msgs = snap.values['messages']
print('=== Latest state for thread alice ===')
print(f'  step        : {snap.metadata["step"]}')
print(f'  checkpoint  : {snap.config["configurable"]["checkpoint_id"][:24]}...')
print(f'  next nodes  : {snap.next or "(done)"}')
print(f'  source      : {snap.metadata["source"]}')
print(f'  messages    : {len(msgs)}')
for i, m in enumerate(msgs, 1):
    print(f'    [{i}] {m.type.upper():<6} {m.content[:60]}')

In [ ]:
history = list(graph.get_state_history(alice))
print(f'Full checkpoint history for alice: {len(history)} checkpoints\n')
rows = []
for snap in history:
    msgs = snap.values.get('messages', [])
    rows.append({
        'step'          : snap.metadata.get('step'),
        'checkpoint_id' : snap.config['configurable'].get('checkpoint_id', '')[:20] + '...',
        'messages'      : len(msgs),
        'next'          : str(snap.next),
        'source'        : snap.metadata.get('source'),
    })
display(pd.DataFrame(rows))

## 5 · All threads in the database

In [ ]:
df_threads = pd.read_sql_query(
    'SELECT thread_id, COUNT(*) as checkpoints, MAX(checkpoint_id) as latest_checkpoint '
    'FROM checkpoints GROUP BY thread_id ORDER BY latest_checkpoint DESC',
    conn
)
print('All threads stored in SQLite:')
display(df_threads)

## Summary

```
SQLite file layout
──────────────────────────────────────────────────────────
 checkpoints          1 row per super-step per thread
                      checkpoint = full state blob (msgpack)
                      metadata   = step, source, writes map

 checkpoint_writes    1 row per channel write within a step
                      used for fault tolerance recovery
                      value = serialised channel value
──────────────────────────────────────────────────────────
```

**Key takeaway:** The raw blobs look like noise, but `get_state()` / `get_state_history()`
decode them into clean Python objects. SQLite is just the storage cabinet — LangGraph
is the librarian who knows how to read and write every shelf.